# 01：解压、读取和初步识别原始数据

本 notebook 完成以下任务：
1. 自动创建项目目录结构
2. 两层自动解压 CSMAR 原始数据
   - 第一层：CSMAR.zip → 提取 7 个子 zip 到 `data/data_raw_zip/`
   - 第二层：每个子 zip 解压到各自子目录 `data/raw/<zip名>/`
3. 读取所有原始数据文件，识别编码、列名、样本量
4. 记录处理日志

In [ ]:
import os, sys, zipfile, glob, shutil
import pandas as pd
from datetime import datetime

PROJECT_ROOT = os.getcwd()
print(f'Project root: {PROJECT_ROOT}')

## 1. 自动创建项目目录结构

In [ ]:
dirs = [
    'data/data_raw_zip', 'data/raw', 'data/dict',
    'data/clean', 'data/combined', 'data/temp',
    'output/tables', 'output/figures',
]
for d in dirs:
    os.makedirs(os.path.join(PROJECT_ROOT, d), exist_ok=True)
    print(f'  Created: {d}')
print('\nDirectory structure created.')
log_lines = []

## 2. 第一层解压：CSMAR.zip → data/data_raw_zip/

CSMAR.zip 内部包含 7 个子 zip。先提取子 zip 到 `data/data_raw_zip/`。

In [ ]:
zip_dir = os.path.join(PROJECT_ROOT, 'data', 'data_raw_zip')
raw_dir = os.path.join(PROJECT_ROOT, 'data', 'raw')

# Search for CSMAR.zip
candidates = [
    os.path.join(PROJECT_ROOT, 'CSMAR.zip'),
    os.path.join(PROJECT_ROOT, 'data', 'CSMAR.zip'),
    os.path.join(PROJECT_ROOT, 'data', 'raw', 'CSMAR.zip'),
    os.path.join(zip_dir, 'CSMAR.zip'),
]
csmar_zip_path = next((p for p in candidates if os.path.exists(p)), None)

if csmar_zip_path is None:
    raise FileNotFoundError(
        'CSMAR.zip not found.\n'
        'Download: https://pan.sysu.edu.cn/link/AAF144310393AF430DAD66672C8254B42D\n'
        'Password: Kdxh\n'
        'Place CSMAR.zip in project root or data/ directory.'
    )

print(f'Found CSMAR.zip: {csmar_zip_path}')

# Extract inner zip files
with zipfile.ZipFile(csmar_zip_path, 'r') as outer_zip:
    inner_zips = [f for f in outer_zip.namelist() if f.endswith('.zip')]
    print(f'\n{len(inner_zips)} inner zip files:')
    for f in inner_zips:
        basename = os.path.basename(f)
        size = outer_zip.getinfo(f).file_size
        print(f'  {basename} ({size/1024/1024:.1f} MB)')
    
    for f in inner_zips:
        basename = os.path.basename(f)
        target = os.path.join(zip_dir, basename)
        with outer_zip.open(f) as src, open(target, 'wb') as dst:
            shutil.copyfileobj(src, dst)
        print(f'  OK: {basename}')
        log_lines.append(f'{datetime.now().isoformat()}: Extracted inner zip {basename}')

print(f'\n{len(os.listdir(zip_dir))} files in data/data_raw_zip/')

## 3. 第二层解压：每个子 zip → data/raw/<zip名>/

因为多个 zip 内含同名文件（如 `跨表查询_沪深京股票(年频).xlsx`），每个 zip 解压到独立子目录，避免相互覆盖。

In [ ]:
# Clean and recreate raw directory
if os.path.exists(raw_dir):
    shutil.rmtree(raw_dir)
os.makedirs(raw_dir, exist_ok=True)

zip_files = sorted(glob.glob(os.path.join(zip_dir, '*.zip')))
print(f'Extracting {len(zip_files)} zips to data/raw/...\n')

for zf in zip_files:
    basename = os.path.basename(zf).replace('.zip', '')
    target_dir = os.path.join(raw_dir, basename)
    os.makedirs(target_dir, exist_ok=True)
    
    try:
        with zipfile.ZipFile(zf, 'r') as z:
            z.extractall(target_dir)
        print(f'  OK: {basename}')
        log_lines.append(f'{datetime.now().isoformat()}: Extracted {basename} to data/raw/{basename}/')
    except Exception as e:
        print(f'  FAIL: {basename} - {e}')
        log_lines.append(f'{datetime.now().isoformat()}: FAILED {basename}: {e}')

print(f'\nExtraction complete.')

## 4. 扫描并识别所有数据文件

In [ ]:
extensions = ['*.xlsx', '*.xls', '*.csv', '*.dta']
raw_files = []
for ext in extensions:
    raw_files.extend(glob.glob(os.path.join(raw_dir, '**', ext), recursive=True))

raw_files = sorted(raw_files)
print(f'Found {len(raw_files)} data files:\n')
for f in raw_files:
    rel = os.path.relpath(f, PROJECT_ROOT)
    size_kb = os.path.getsize(f) / 1024
    print(f'  {rel} ({size_kb:.1f} KB)')

## 5. 快速识别每个文件的结构

In [ ]:
file_info = []

for f in raw_files:
    basename = os.path.basename(f)
    rel = os.path.relpath(f, PROJECT_ROOT)
    parent = os.path.basename(os.path.dirname(f))
    
    info = {'filepath': rel, 'source_zip': parent, 'filename': basename,
            'status': 'unknown', 'rows': 0, 'cols': 0, 'sample_columns': ''}
    
    try:
        df = pd.read_excel(f, nrows=3)
        info['status'] = 'ok'
        info['cols'] = len(df.columns)
        info['sample_columns'] = str(df.columns.tolist()[:25])
        
        # Read full to count rows
        df_full = pd.read_excel(f)
        info['rows'] = len(df_full)
        years = df_full['EndDate'].dropna().unique() if 'EndDate' in df_full.columns else []
        years_str = ', '.join(sorted([str(y) for y in years if str(y).isdigit()])[:5])
        print(f'  OK: {rel}')
        print(f'       {info["rows"]} rows x {info["cols"]} cols | years: {years_str}...')
    except Exception as e:
        info['status'] = 'failed'
        print(f'  FAIL: {rel} - {e}')
    
    file_info.append(info)

# Save overview
overview = pd.DataFrame(file_info)
overview.to_csv(os.path.join(PROJECT_ROOT, 'data', 'temp', 'file_overview.csv'), index=False)
print(f'\nFile overview saved. {sum(1 for x in file_info if x["status"]=="ok")}/{len(file_info)} files OK.')

## 6. 原始文件清单

| 文件 | 来源 zip | 行数 | 列数 | 内容 |
|------|----------|------|------|------|
| 跨表查询_沪深京股票(年频).xlsx | 资产负债表-2000-2010 | 64,166 | 32 | 资产负债表 (2000-2010) |
| 跨表查询_沪深京股票(年频).xlsx | 资产负债表-2011-2024 | 81,665 | 32 | 资产负债表 (2011-2024) |
| 跨表查询_沪深京股票(年频).xlsx | 利润表-现金流量表-2000-2010 | 64,166 | 36 | 利润表+现金流量表 (2000-2010) |
| 跨表查询_沪深京股票(年频).xlsx | 利润表-现金流量表-2011-2024 | 81,665 | 36 | 利润表+现金流量表 (2011-2024) |
| 常用变量查询（年度）.xlsx | CSMAR常用变量-2000-2024 | ? | 33 | 股权结构、治理变量 |
| STK_LISTEDCOINFOANL.xlsx | 上市公司基本信息年度表 | ? | 40 | 公司信息、行业分类 |
| STK_LISTEDCOINFOCHG.xlsx | 上市公司基本信息变更表 | ? | 8 | 公司信息变更（本作业不使用） |

## 7. 写入处理日志

In [ ]:
log_lines.append(f'{datetime.now().isoformat()}: 01_extract_raw_data complete - {len(raw_files)} data files')

log_path = os.path.join(PROJECT_ROOT, 'process_log.txt')
with open(log_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(log_lines) + '\n')
print(f'Process log written to {log_path}')